In [1]:
import json
import os
import logging

from IPython.display import clear_output
from tqdm import tqdm

from src import comp_models, Gomoku, ADP_Dense_Player, Player

DIR_PATH = "_dens2"

RESULTS_PATH = os.path.join(DIR_PATH, "logs/results.log")

# log to file leaderboard_dens2.jsonl
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
fh = logging.FileHandler(RESULTS_PATH)
fh.setLevel(logging.INFO)
logger.addHandler(fh)

In [5]:
def my_print(*args, **kwargs):
    clear_output(wait=True)
    print(*args, **kwargs)

def play_game(game: Gomoku, player1: Player, player2: Player, verbose: int = 0):
    while not game.fin():
        move = player1.next_move(game)
        game.play(move)
        if verbose > 0:
            if verbose > 1:
                print(player1.next_move_probs(game))
            print(game)
        if game.fin():
            break
        move = player2.next_move(game)
        game.play()
        if verbose > 0:
            if verbose > 1:
                print(player2.next_move_probs(game))
            print(game)
    return game.winner

def play_n_games(game: Gomoku, player1: Player, player2: Player, n: int, verbose: int = 0):
    avg = 0
    avgs = []
    for i in range(n):
        score = (play_game(game.copy(), player1, player2, verbose) + 1) / 2
        avg *= i / (i + 1)
        avg += score / (i + 1)
        avgs += [avg]
    return avgs

def tournament(game_kwargs, models: list[Player], n_test_games: int, start_ind: int = 0) -> list[tuple[int, int, int, int]]:
    total_ind = 0
    with tqdm(
        total=n_test_games * len(models) * (len(models) - 1) // 2, 
        position=0, 
        leave=False, 
        desc="Tournament"
    ) as bar:
        for i in range(len(models)):
            for j in range(i+1, len(models)):
                if total_ind < start_ind:
                    bar.update(n_test_games)
                    total_ind += n_test_games
                    continue
                n_wins, l_history = 0, 0
                for _ in range(n_test_games):
                    win, starts, len_history = comp_models(game_kwargs, models[i], models[j])
                    n_wins += (win > 0) == starts
                    l_history += len_history
                    bar.update(1)
                    total_ind += 1
                n_wins, l_history = n_wins / n_test_games, l_history / n_test_games
                yield (i, j, n_wins, l_history)

In [ ]:
game_kwargs = {
    'M': 8,
    'N': 8,
    'K': 5,
    'ADJ': 2,
}

adp_args = {
    "alpha": 0.9,
    "magnify": 1,
    "gamma": 0.9,
    "lr": 0.01,
    "n_steps": 1,
    "epsilon": 0.1,
    "logger": logger, 
    # "M": game_kwargs['M'],
    # "N": game_kwargs['N'],
}

players = [
    ADP_Dense_Player(model_path=os.path.join(DIR_PATH, f"models/epoch_{i}.h5"), **adp_args)
    for i in range(250, 5050, 250)
]

for (i, j, n_wins, l_history) in tournament(game_kwargs, players, n_test_games=25, start_ind=2300):
    logger.info(json.dumps({
        "first": (i + 1) * 250, 
        "second": (j + 1) * 250, 
        "avg_win": n_wins, 
        "avg_len": l_history
    }))